# Siamese Network

In [ ]:
# import the necessary packages
import tensorflow as tf
# import keras
import numpy as np
import cv2
import os
import glob
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from euclideanDist import EuclideanDistanceLayer
import random


In [ ]:
# --- 1. Data Loading and Basic Preparation ---
# (Keep your existing path definitions and file loading logic here)
# ... (folder_A, folder_B, image_extensions, load_and_preprocess_image, etc.) ...

# --- Define Paths ---
folder_A = "images/Top100/512Top100"
folder_B = "images/Top100/512Top100frame"
image_extensions = ('.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff')
img_shape = (128, 128, 3) # Keep consistent (H, W, C)
K = tf.keras.backend

# --- Helper function to load and preprocess images ---
def load_and_preprocess_image(image_path, target_shape=img_shape):
    target_size = (target_shape[0], target_shape[1])
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"Warning: Could not read image {image_path}")
        return None
    img = cv2.resize(img, target_size)
    img = img.astype("float32") / 255.0 # Normalize BEFORE augmentation
    img = np.expand_dims(img, axis=-1)
    img_3channel = np.concatenate([img, img, img], axis=-1)
    # --- End Conversion ---

    # Ensure the final shape matches the target shape
    if img_3channel.shape != target_shape:
        print(f"Error: Final shape {img_3channel.shape} doesn't match target {target_shape} for {image_path}")
        # Handle error appropriately, maybe return None or raise exception
        return None

    return img_3channel

def shuffle_lists(l1, l2, l3, seed=None):
    if not (len(l1) == len(l2) == len(l3)):
        print("Error: Input lists must have the same length.")
        return None, None, None

    if not l1: # Handle empty lists
        return [], [], []

    # Combine the lists into tuples
    combined = list(zip(l1, l2, l3))

    # Set seed for reproducibility if provided *before* shuffling
    if seed is not None:
        print(f"Using random seed: {seed}")
        random.seed(seed)

    # Shuffle the combined list in-place
    random.shuffle(combined)

    # Unzip back into separate sequences (these are tuples)
    s1, s2, s3 = zip(*combined)

    # Convert back to lists and return
    return list(s1), list(s2), list(s3)

# --- Generate file lists and labels (Same as before) ---
path_pattern_A = os.path.join(folder_A, '*')
files_A_orig = sorted([f for f in glob.glob(path_pattern_A)
                  if os.path.isfile(f) and f.lower().endswith(image_extensions)])

path_pattern_B = os.path.join(folder_B, '*')
files_B_orig = sorted([f for f in glob.glob(path_pattern_B)
                  if os.path.isfile(f) and f.lower().endswith(image_extensions)])

num_matches = min(100, len(files_A_orig), len(files_B_orig))
files_A_match = files_A_orig[:num_matches]
files_B_match = files_B_orig[:num_matches]
labels_match = [1.0] * num_matches

if len(files_A_orig) > 1 and len(files_B_orig) >= num_matches:
    files_A_skewed = files_A_orig[1:num_matches+1]
    if len(files_A_skewed) < num_matches:
         files_A_skewed.append(files_A_orig[0])
    files_B_nonmatch = files_B_orig[:num_matches]
    labels_nonmatch = [0.0] * num_matches
else:
    files_A_skewed = []
    files_B_nonmatch = []
    labels_nonmatch = []
    print("Warning: Not enough files for non-matching pairs.")

all_files_A = files_A_match + files_A_skewed
all_files_B = files_B_match + files_B_nonmatch
all_labels = np.array(labels_match + labels_nonmatch) # Labels first

# --- Load paths, not images directly (for tf.data) ---
pairs_paths_A = []
pairs_paths_B = []
valid_labels = []

print(f"Checking {len(all_files_A)} image pair paths...")
for i in range(len(all_files_A)):
    # Basic check if files exist before adding paths
    if os.path.exists(all_files_A[i]) and os.path.exists(all_files_B[i]):
        pairs_paths_A.append(all_files_A[i])
        pairs_paths_B.append(all_files_B[i])
        valid_labels.append(all_labels[i])
    else:
        print(f"Skipping pair due to missing file: {all_files_A[i]} or {all_files_B[i]}")

pairs_paths_A, pairs_paths_B, valid_labels = shuffle_lists(pairs_paths_A, pairs_paths_B, valid_labels, seed=42)

all_labels = np.array(valid_labels) # Update labels to match valid pairs
print(f"Found {len(pairs_paths_A)} valid pairs.")

# --- Split paths and labels into Training and Validation Sets ---
if len(pairs_paths_A) > 0:
    indices = np.arange(len(pairs_paths_A))
    (train_idx, val_idx) = train_test_split(indices, test_size=0.2, random_state=42, stratify=all_labels)

    # Split paths first
    train_paths_A, val_paths_A = np.array(pairs_paths_A)[train_idx], np.array(pairs_paths_A)[val_idx]
    train_paths_B, val_paths_B = np.array(pairs_paths_B)[train_idx], np.array(pairs_paths_B)[val_idx]
    y_train, y_val = all_labels[train_idx], all_labels[val_idx]

    print(f"Training pairs: {len(train_paths_A)}, Validation pairs: {len(val_paths_A)}")
else:
    print("Cannot proceed without valid data paths.")
    exit()

# --- 2. Define Data Augmentation ---

# Using tf.keras layers for simplicity within map function
data_augmentation = tf.keras.Sequential([
  tf.keras.layers.RandomFlip("horizontal"),
  tf.keras.layers.RandomRotation(0.1), # rotation range
  tf.keras.layers.RandomZoom(0.1), # zoom range
  # Add brightness/contrast for grayscale robustness
  # Note: These expect channels last, even for grayscale
  tf.keras.layers.RandomBrightness(factor=0.1),
  tf.keras.layers.RandomContrast(factor=0.1),
  # Add more aggressive augmentation if needed (e.g., higher factors)
], name='data_augmentation')

# --- tf.data Pipeline ---
def load_and_augment_pair(path_a, path_b, label):
    # Define inner function to handle py_function compatibility
    def _load_and_process(p_a, p_b):
        img_a = load_and_preprocess_image(p_a.numpy().decode('utf-8'), target_shape=img_shape)
        img_b = load_and_preprocess_image(p_b.numpy().decode('utf-8'), target_shape=img_shape)
        # Handle potential loading errors inside map
        if img_a is None: img_a = np.zeros(img_shape, dtype=np.float32)
        if img_b is None: img_b = np.zeros(img_shape, dtype=np.float32)
        return img_a, img_b

    # Use tf.py_function to wrap the OpenCV loading logic
    img_a, img_b = tf.py_function(
        _load_and_process,
        [path_a, path_b],
        [tf.float32, tf.float32]
    )
    # Ensure shapes are set correctly after py_function
    img_a.set_shape(img_shape)
    img_b.set_shape(img_shape)

    # Apply the SAME augmentation to both images in the pair? Often yes for geometric.
    # Or apply DIFFERENTLY? Depends on what you want to be robust to.
    # Let's apply independently here for more variety:
    img_a_aug = data_augmentation(img_a, training=True)
    img_b_aug = data_augmentation(img_b, training=True)

    return (img_a_aug, img_b_aug), label # Return tuple (inputs), label

def load_pair(path_a, path_b, label):
    # Version without augmentation for validation/test set
    def _load_and_process(p_a, p_b):
        img_a = load_and_preprocess_image(p_a.numpy().decode('utf-8'), target_shape=img_shape)
        img_b = load_and_preprocess_image(p_b.numpy().decode('utf-8'), target_shape=img_shape)
        if img_a is None: img_a = np.zeros(img_shape, dtype=np.float32)
        if img_b is None: img_b = np.zeros(img_shape, dtype=np.float32)
        return img_a, img_b

    img_a, img_b = tf.py_function(
        _load_and_process,
        [path_a, path_b],
        [tf.float32, tf.float32]
    )
    img_a.set_shape(img_shape)
    img_b.set_shape(img_shape)
    return (img_a, img_b), label

AUTOTUNE = tf.data.AUTOTUNE
batch_size = 32 # Keep consistent

train_ds = tf.data.Dataset.from_tensor_slices((train_paths_A, train_paths_B, y_train))
train_ds = train_ds.shuffle(len(train_paths_A)) # Shuffle paths
train_ds = train_ds.map(load_and_augment_pair, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.batch(batch_size)
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE) # Optimizes performance

val_ds = tf.data.Dataset.from_tensor_slices((val_paths_A, val_paths_B, y_val))
val_ds = val_ds.map(load_pair, num_parallel_calls=AUTOTUNE) # No augmentation for validation
val_ds = val_ds.batch(batch_size)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)


# --- 3. Define the CNN Backbone with Regularization ---

def build_base_network(input_shape, embedding_dim=128, regularization_factor=0.001):
    """Builds the CNN model with Dropout and L2 regularization."""
    input = tf.keras.layers.Input(shape=input_shape, name="base_input")

    # Add Batch Normalization potentially
    # x = BatchNormalization()(input) # Optional: Normalize input batch

    x = tf.keras.layers.Conv2D(64, (7, 7), padding="same", activation="relu",
               kernel_regularizer=tf.keras.regularizers.l2(regularization_factor), name="conv1")(input) # Regularizer added
    # x = BatchNormalization()(x) # Optional
    x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2), name="pool1")(x)
    x = tf.keras.layers.Dropout(0.2)(x) # Dropout added

    x = tf.keras.layers.Conv2D(128, (5, 5), padding="same", activation="relu",
               kernel_regularizer=tf.keras.regularizers.l2(regularization_factor), name="conv2")(x) # Regularizer added
    # x = BatchNormalization()(x) # Optional
    x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2), name="pool2")(x)
    x = tf.keras.layers.Dropout(0.2)(x) # Dropout added

    x = tf.keras.layers.Conv2D(256, (3, 3), padding="same", activation="relu",
               kernel_regularizer=tf.keras.regularizers.l2(regularization_factor), name="conv3")(x) # Regularizer added
    # x = BatchNormalization()(x) # Optional
    x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2), name="pool3")(x)
    x = tf.keras.layers.Dropout(0.2)(x) # Dropout added

    x = tf.keras.layers.Flatten(name="flatten")(x)
    x = tf.keras.layers.Dropout(0.4)(x) # Higher dropout before final dense layer

    # Dense layer to produce the embedding
    x = tf.keras.layers.Dense(embedding_dim, activation=None, # Or None/Linear activation? Test both.
              kernel_regularizer=tf.keras.regularizers.l2(regularization_factor), name="dense_embedding")(x) # Regularizer added

    model = tf.keras.models.Model(inputs=input, outputs=x, name="BaseNetwork")
    return model

def build_pretrained_base_network(input_shape, embedding_dim=128):
    """
    Builds a base network using a pre-trained MobileNetV2 model.
    Freezes the base model layers and adds a custom head for embedding.
    """
    # Load MobileNetV2 pre-trained on ImageNet, excluding the top classification layer
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=input_shape,
        include_top=False,  # Important: Remove the original classifier
        weights='imagenet' # Use pre-trained weights
    )

    # --- Freeze the pre-trained layers ---
    # We don't want to update ImageNet weights initially, just train our new layers
    base_model.trainable = True

    # Define the input matching the expected shape
    inputs = tf.keras.layers.Input(shape=input_shape, name="base_input")

    # Pass input through the base model
    # Set training=False so Batch Norm layers run in inference mode
    x = base_model(inputs, training=False)

    # Add pooling to flatten feature maps
    x = tf.keras.layers.GlobalAveragePooling2D(name="global_avg_pool")(x)

    # Optional: Add dropout for regularization
    x = tf.keras.layers.Dropout(0.3, name="head_dropout")(x) # Adjust rate if needed

    # --- Final Embedding Layer ---
    # Use linear activation (None) and L2 normalization
    x = tf.keras.layers.Dense(embedding_dim, activation=None, name="dense_embedding")(x)
    outputs = tf.keras.layers.UnitNormalization(axis=1, name='l2_normalization')(x)

    # Create the new base network model
    model = tf.keras.models.Model(inputs=inputs, outputs=outputs, name="PretrainedBaseNetwork")
    return model

# --- 4. Build the Siamese Architecture (Same as before) ---

input_shape = img_shape

embedding_dim = 128 # Or try 64 if overfitting persists
l2_reg = 0.001 # L2 factor, tune if needed

input_A = tf.keras.layers.Input(shape=input_shape, name="input_A")
input_B = tf.keras.layers.Input(shape=input_shape, name="input_B")

# base_network = build_base_network(input_shape, embedding_dim, l2_reg)
base_network = build_pretrained_base_network(input_shape, embedding_dim)
base_network.summary()

processed_A = base_network(input_A)
processed_B = base_network(input_B)

# Euclidean distance function (Same as before)
def euclidean_distance(vectors):
    (featsA, featsB) = vectors
    sumSquared = K.sum(K.square(featsA - featsB), axis=1, keepdims=True)
    return K.sqrt(K.maximum(sumSquared, K.epsilon()))

distance_output = EuclideanDistanceLayer(name="euclidean_distance")([processed_A, processed_B])

model = tf.keras.models.Model(inputs=[input_A, input_B], outputs=distance_output, name="SiameseNetwork")
model.summary()

# --- 5. Define Contrastive Loss (Same as before) ---

def contrastive_loss(y_true, y_pred_dist, margin=1.0):
    y_true = tf.cast(y_true, tf.float32)
    square_pred = K.square(y_pred_dist)
    margin_square = K.square(K.maximum(margin - y_pred_dist, 0))
    # Add a small epsilon to the loss components for numerical stability? Sometimes helps.
    # loss = y_true * square_pred + (1 - y_true) * margin_square
    # return K.mean(loss + K.epsilon())
    return K.mean(y_true * square_pred + (1 - y_true) * margin_square)

# --- 6. Compile and Train with Callbacks ---

print("Compiling model...")
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001) # Keep learning rate potentially small
model.compile(loss=contrastive_loss, optimizer=optimizer)

# Define Callbacks
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',    # Monitor validation loss
    patience=10,           # Stop after 10 epochs with no improvement
    restore_best_weights=True, # Restore weights from the best epoch
    verbose=1
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,           # Reduce LR by a factor of 5
    patience=5,           # Reduce LR after 5 epochs with no improvement
    min_lr=1e-6,          # Minimum learning rate
    verbose=1
)



print("Starting training with augmentation and callbacks...")
epochs = 50 # Can set higher now, early stopping will handle it


history = model.fit(
    train_ds, # Use the tf.data dataset
    validation_data=val_ds, # Use the validation dataset
    epochs=epochs,
    callbacks=[early_stopping, reduce_lr] # Add callbacks
)

print("Training complete.")

# --- Optional: Evaluate or Plot History (Same as before) ---
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(loc='upper right')
plt.ylim(bottom=0) # Often useful to set y-axis minimum to 0 for loss plots
plt.grid(True)
plt.show()

# --- Save the final model (potentially the best one due to EarlyStopping) ---
model.save("siamese_artwork_model_augmented.keras")



In [ ]:
K = tf.keras.backend


# --- Configuration (MUST MATCH TRAINING) ---
IMG_HEIGHT = 128
IMG_WIDTH = 128
# Assuming Grayscale (1 channel). If you used color, change to 3.
IMG_CHANNELS = 3
IMG_SHAPE = (IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)
MODEL_PATH = "siamese_artwork_model_augmented.keras" # Your saved model file

# --- Define Custom Objects Used During Training ---
# You NEED to redefine the custom loss function used during training
# so Keras can load the model.


def contrastive_loss(y_true, y_pred_dist, margin=1.0):
    y_true = tf.cast(y_true, tf.float32)
    square_pred = K.square(y_pred_dist)
    margin_square = K.square(K.maximum(margin - y_pred_dist, 0))
    return K.mean(y_true * square_pred + (1 - y_true) * margin_square)

# def euclidean_distance(vectors):
#     (featsA, featsB) = vectors
#     sumSquared = K.sum(K.square(featsA - featsB), axis=1, keepdims=True)
#     return K.sqrt(K.maximum(sumSquared, K.epsilon()))

# # Add output_shape=(1,) to the Lambda layer definition
# distance_euclidean = tf.keras.layers.Lambda(
#     euclidean_distance,
#     name="euclidean_distance",
#     output_shape=(1,) # <--- ADD THIS ARGUMENT
# )([processed_A, processed_B])

# --- Helper function to load and preprocess images ---
def load_and_preprocess_image(image_path, target_shape=IMG_SHAPE):
    target_size = (target_shape[0], target_shape[1])
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"Warning: Could not read image {image_path}")
        return None
    img = cv2.resize(img, target_size)
    img = img.astype("float32") / 255.0 # Normalize BEFORE augmentation
    img = np.expand_dims(img, axis=-1)
    img_3channel = np.concatenate([img, img, img], axis=-1)
    # --- End Conversion ---

    # Ensure the final shape matches the target shape
    if img_3channel.shape != target_shape:
        print(f"Error: Final shape {img_3channel.shape} doesn't match target {target_shape} for {image_path}")
        # Handle error appropriately, maybe return None or raise exception
        return None

    return img_3channel

# --- Helper function to load and preprocess a single image ---
def load_and_preprocess_inference_image(image_path, target_shape=IMG_SHAPE):
    """Loads grayscale, preprocesses, and converts to 3 channels for inference."""
    target_size = (target_shape[0], target_shape[1])

    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE) # Load grayscale first
    if img is None:
        print(f"Error: cv2.imread failed for path: {image_path}")
        return None
    img = cv2.resize(img, target_size)
    img = img.astype("float32") / 255.0
    # Convert grayscale to 3 channels
    img = np.expand_dims(img, axis=-1) # Add channel axis -> (H, W, 1)
    img_3channel = np.concatenate([img, img, img], axis=-1) # Stack -> (H, W, 3)

    if img_3channel.shape != target_shape:
         print(f"Error: Final shape {img_3channel.shape} doesn't match target {target_shape} for {image_path}")
         return None
    return img_3channel

custom_objects = {
    "contrastive_loss": contrastive_loss,
    "EuclideanDistanceLayer": EuclideanDistanceLayer
}

# ---

print(f"Loading model from {MODEL_PATH}...")
model = None # Initialize model variable
try:
    model = tf.keras.models.load_model(MODEL_PATH, custom_objects=custom_objects)
    print("Model loaded successfully.")
    model.summary() # Optional: print model summary
except Exception as e:
    print(f"Error loading model: {e}")
    # Handle error, maybe exit() or raise



# --- Define the image paths ---

folder_A = "images/Top100/512Top100"
folder_B = "images/Top100/512Top100Frame"

image_extensions = ('.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff')

path_pattern_A = os.path.join(folder_A, '*')
files_A = [f for f in glob.glob(path_pattern_A)
           if os.path.isfile(f) and f.lower().endswith(image_extensions)]

path_pattern_B = os.path.join(folder_B, '*')
files_B = [f for f in glob.glob(path_pattern_B)
           if os.path.isfile(f) and f.lower().endswith(image_extensions)]

files_A.sort()
files_B.sort()

files_A_skewd = files_A.copy()
first_element = files_A_skewd.pop(0)
files_A_skewd.append(first_element)

# print(files_A)
# print(files_A_skewd)

files_A = files_A + files_A_skewd
files_B = files_B + files_B

print(len(files_A))
print(len(files_B))

res = []
tru = [True] * 100 + [False] * 100

# image_path_1 = "images/test/img512.png"
# image_path_2 = "images/test/imgFrame512.png"

for i in range(len(files_A)):

    image_path_1 = files_A[i]
    image_path_2 = files_B[i]

    # THE PROBLEM IS HERE! DO WE NEED TO ADD BATCH DIMENSION? WHY "IF IMAGE IS NOT NONE"?

    # --- Preprocess the images ---
    print("Preprocessing images...")
    img1_processed = load_and_preprocess_image(image_path_1, target_shape=IMG_SHAPE)
    img2_processed = load_and_preprocess_image(image_path_2, target_shape=IMG_SHAPE)


    # Add batch dimension to the images
    img1_processed = np.expand_dims(img1_processed, axis=0)
    img2_processed = np.expand_dims(img2_processed, axis=0)

    # --- Predict the distance ---
    print("Predicting distance...")
    # The model expects a list/tuple of the two inputs
    predicted_distance = model.predict([img1_processed, img2_processed])[0][0]

    print(f"\nPredicted distance between the two images: {predicted_distance:.4f}")

    # --- Compare to threshold ---
    # !!! IMPORTANT: You MUST determine this threshold based on your validation set !!!
    # Common starting points might be 0.5 * margin, but tune it rigorously.
    # Let's use an EXAMPLE threshold:
    CHOSEN_THRESHOLD = 0.6 # <--- TUNE THIS VALUE BASED ON VALIDATION RESULTS

    print(f"Using similarity threshold: {CHOSEN_THRESHOLD}")

    if predicted_distance < CHOSEN_THRESHOLD:
        print("Result: The images are LIKELY the SAME artwork.")
        match = True
    else:
        print("Result: The images are LIKELY DIFFERENT artworks.")
        match = False


    # --- Display the images being compared using Matplotlib ---
    print("Displaying images using Matplotlib...")
    try:
        # Load images using OpenCV
        img1_display = cv2.imread(image_path_1)
        img2_display = cv2.imread(image_path_2)

        # Check if images were loaded successfully
        if img1_display is None:
            raise ValueError(f"Failed to load image: {image_path_1}")
        if img2_display is None:
            raise ValueError(f"Failed to load image: {image_path_2}")

        # Resize images for display
        img1_display = cv2.resize(img1_display, (256, 256))
        img2_display = cv2.resize(img2_display, (256, 256))

        # --- Convert BGR to RGB for Matplotlib display ---
        # Check if the image has 3 channels (is color) before converting
        if len(img1_display.shape) == 3 and img1_display.shape[2] == 3:
            img1_rgb = cv2.cvtColor(img1_display, cv2.COLOR_BGR2RGB)
        else:
            img1_rgb = img1_display # Keep as is if grayscale

        if len(img2_display.shape) == 3 and img2_display.shape[2] == 3:
            img2_rgb = cv2.cvtColor(img2_display, cv2.COLOR_BGR2RGB)
        else:
            img2_rgb = img2_display # Keep as is if grayscale

        # Stack images horizontally for comparison
        combined_display = np.hstack([img1_rgb, img2_rgb])

        # --- Create the plot ---
        plt.figure(figsize=(10, 5)) # Adjust figure size as needed

        # Display the combined image
        plt.imshow(combined_display)

        # Determine title color and text based on match status
        title_color = 'green' if match else 'red'
        title_text = f"Match? {match} | Distance: {predicted_distance:.3f} (Threshold: {CHOSEN_THRESHOLD})"
        # title_color = 'black'
        # title_text = "Match?"


        # Add title with status and distance
        plt.title(title_text, color=title_color)

        # Hide axes for a cleaner look
        plt.axis('off')

        # Show the plot
        plt.show()

    except Exception as e:
        # Print specific error message from the exception
        print(f"Could not display images using Matplotlib: {e}")

    res.append(match)



In [ ]:
# 4. Initialize counters
true_positives = 0
true_negatives = 0
false_positives = 0
false_negatives = 0

# 5. Compare lists element by element
for i in range(len(tru)):
    actual = tru[i]
    predicted = res[i]

    if actual is True and predicted is True:
        true_positives += 1
    elif actual is False and predicted is False:
        true_negatives += 1
    elif actual is False and predicted is True:
        # Actual is Negative, but predicted Positive -> False Positive
        false_positives += 1
    elif actual is True and predicted is False:
        # Actual is Positive, but predicted Negative -> False Negative
        false_negatives += 1

# 6. Print the results (Confusion Matrix Components)
print("\n--- Confusion Matrix Components ---")
print(f"True Positives (TP):  {true_positives}")
print(f"True Negatives (TN):  {true_negatives}")
print(f"False Positives (FP): {false_positives} (Type I Error)")
print(f"False Negatives (FN): {false_negatives} (Type II Error)")

# 7. Verification (Optional but recommended)
total_calculated = true_positives + true_negatives + false_positives + false_negatives
print(f"\nTotal items calculated: {total_calculated}")
print(f"Original list length:   {len(tru)}")

# 8. Calculate derived metrics (Optional)
print("\n--- Derived Metrics ---")
total_population = total_calculated

# Accuracy: (TP + TN) / Total
accuracy = (true_positives + true_negatives) / total_population if total_population > 0 else 0
print(f"Accuracy: {accuracy:.4f}")

# Precision: TP / (TP + FP) - How many selected positives are actually positive?
precision_denominator = true_positives + false_positives
precision = true_positives / precision_denominator if precision_denominator > 0 else 0
print(f"Precision: {precision:.4f}")

# Recall (Sensitivity): TP / (TP + FN) - How many actual positives were found?
recall_denominator = true_positives + false_negatives
recall = true_positives / recall_denominator if recall_denominator > 0 else 0
print(f"Recall (Sensitivity): {recall:.4f}")

# Specificity: TN / (TN + FP) - How many actual negatives were correctly identified?
specificity_denominator = true_negatives + false_positives
specificity = true_negatives / specificity_denominator if specificity_denominator > 0 else 0
print(f"Specificity: {specificity:.4f}")

# F1 Score: 2 * (Precision * Recall) / (Precision + Recall)
f1_denominator = precision + recall
f1_score = 2 * (precision * recall) / f1_denominator if f1_denominator > 0 else 0
print(f"F1 Score: {f1_score:.4f}")